# Hands-on with Real LLM Sampling Parameters

## Part 1: Greedy Decoding with `top_k=1`

The reading introduced greedy decoding and sampling. We will begin with the greedy approach using a real model: ask `qwen3.5:2b` to finish the same sentence ten times while holding the prompt and decoding parameters constant. Each completion will use a different seed.

Setting `top_k=1` keeps only the model's highest-scoring token at every step. With only one candidate left, the pseudorandom draw has no choice to make, so changing the seed should not change the completion. Although `temperature` and `top_p` are still shown and passed to Ollama, they cannot restore candidates already removed by `top_k`.

We pass every parameter explicitly so the experiment records its settings. As in Session 1, we also disable Qwen's thinking mode so it does not introduce another behavior into the comparison.

In [ ]:
import ollama

MODEL = "qwen3.5:2b"
SYSTEM_PROMPT = (
    "Your task is to complete the sentence starting from the user's words. "
    "Respond starting with the user's words to begin the sentence you are completing. "
    "Keep your response simple and to the point."
)
USER_PROMPT = "The cat "
N_COMPLETIONS = 10
FIRST_SEED = 1
MAX_GENERATED_TOKENS = 64

GREEDY_PARAMETERS = {
    "top_k": 1,
    "temperature": 1.0,
    "top_p": 1.00,
}

print(f"Model: {MODEL}")
print(f"System prompt: {SYSTEM_PROMPT!r}")
print(f"User prompt: {USER_PROMPT!r}")
print(f"Completions: {N_COMPLETIONS}")
print(f"Seeds: {FIRST_SEED} through {FIRST_SEED + N_COMPLETIONS - 1}")
print(f"Sampling parameters: {GREEDY_PARAMETERS}")

Before running the next cell, predict what will happen. Each seed produces a different deterministic sequence of pseudorandom draws, but `top_k=1` leaves only one token to select at each step. How many distinct completions do you expect to see?

In [ ]:
def generate_completions(
    user_prompt: str,
    *,
    top_k: int,
    temperature: float,
    top_p: float,
    expected_prefix: str | None = None,
) -> list[dict]:
    results = []
    earlier_completions = set()

    for completion_number in range(1, N_COMPLETIONS + 1):
        seed = FIRST_SEED + completion_number - 1
        response = ollama.generate(
            model=MODEL,
            prompt=user_prompt,
            system=SYSTEM_PROMPT,
            think=False,
            options={
                "seed": seed,
                "num_predict": MAX_GENERATED_TOKENS,
                "top_k": top_k,
                "temperature": temperature,
                "top_p": top_p,
            },
        )

        completion = response.response.strip()
        result = {
            "completion_number": completion_number,
            "seed": seed,
            "response": completion,
            "generated_tokens": response.eval_count or 0,
            "duration_seconds": (response.total_duration or 0) / 1_000_000_000,
        }

        if expected_prefix is None:
            print(f"{seed}: {completion!r}")
        else:
            evaluation = evaluate_completion(
                completion,
                expected_prefix=expected_prefix,
                is_replica=completion in earlier_completions,
            )
            result.update(evaluation)
            earlier_completions.add(completion)
            print_evaluation_result(result)

        results.append(result)

    return results


def report_diversity(results: list[dict]) -> None:
    unique_responses = {result["response"] for result in results}
    print(f"Unique responses: {len(unique_responses)} of {len(results)}")


greedy_results = generate_completions(USER_PROMPT, **GREEDY_PARAMETERS)
report_diversity(greedy_results)

### Pause and Reflect

- Did the result match your prediction?
- Why did different pseudorandom draws have no effect here?
- If you rerun the cell without changing anything, do the ten responses repeat?

Seeds `1` through `10` make the experiment reproducible. Next we will define a test for whether a completion follows the task, then apply it when we allow more than one candidate to survive.

## Part 2: Define and Apply an Evaluation

Part 1 measured **diversity** by counting unique responses. That does not tell us whether the responses followed the task. To measure task success, we need to translate our expectations into checks that Python can apply consistently to every completion.

Recall the specific task: continue the user's starter, begin the response with those words, and keep the completion simple and to the point. Here the starter is `The cat `. For this experiment, we will call a completion **successful** when all four conditions hold:

1. It begins with an expected prefix, ignoring capitalization.
2. It ends with a period or exclamation point.
3. It contains no more than one period or exclamation point in total.
4. It is not an exact replica of an earlier completion in the same batch.

These criteria are purpose-built for comparing sampling settings on this task. The prefix check measures whether the model followed the continuation instruction. The punctuation checks use one complete sentence as a concrete proxy for "simple and to the point." The replica check makes the batch score sensitive to whether a sampling configuration produces varied useful answers instead of repeatedly producing one valid answer. We still report the distinct-response count separately so that diversity remains visible rather than being confused with formatting quality.

The second and third checks together require exactly one period or exclamation point, but keeping them separate makes a failure easier to diagnose. Both marks are counted literally, so a period used in an abbreviation also counts. Replica detection compares the exact completion text after removing surrounding whitespace; the first occurrence can pass, and later copies fail. Thus, ten identical otherwise-valid completions produce one success out of ten. The evaluator accepts `expected_prefix` as a parameter rather than building `The cat` into the function, which lets us reuse it with another sentence starter. Surrounding whitespace on the completion and expected prefix is ignored.

In [ ]:
def evaluate_completion(
    completion: str,
    *,
    expected_prefix: str,
    is_replica: bool = False,
) -> dict:
    text = completion.strip()
    prefix = expected_prefix.strip()

    if not prefix:
        raise ValueError("expected_prefix must not be empty")

    starts_with_expected_prefix = text.casefold().startswith(
        prefix.casefold()
    )
    ends_with_allowed_punctuation = text.endswith((".", "!"))
    allowed_punctuation_count = text.count(".") + text.count("!")
    at_most_one_allowed_punctuation_mark = allowed_punctuation_count <= 1

    return {
        "starts_with_expected_prefix": starts_with_expected_prefix,
        "ends_with_allowed_punctuation": ends_with_allowed_punctuation,
        "at_most_one_allowed_punctuation_mark": (
            at_most_one_allowed_punctuation_mark
        ),
        "is_replica": is_replica,
        "successful": (
            starts_with_expected_prefix
            and ends_with_allowed_punctuation
            and at_most_one_allowed_punctuation_mark
            and not is_replica
        ),
    }


def evaluate_completions(
    results: list[dict],
    *,
    expected_prefix: str,
) -> list[dict]:
    evaluated_results = []
    earlier_completions = set()

    for result in results:
        completion = result["response"].strip()
        evaluation = evaluate_completion(
            completion,
            expected_prefix=expected_prefix,
            is_replica=completion in earlier_completions,
        )
        evaluated_results.append({**result, **evaluation})
        earlier_completions.add(completion)

    return evaluated_results


def failed_evaluation_checks(result: dict) -> list[str]:
    check_labels = {
        "starts_with_expected_prefix": "expected prefix",
        "ends_with_allowed_punctuation": (
            "ends with period or exclamation point"
        ),
        "at_most_one_allowed_punctuation_mark": (
            "at most one period or exclamation point"
        ),
    }
    failed_checks = [
        label
        for check, label in check_labels.items()
        if not result[check]
    ]
    if result["is_replica"]:
        failed_checks.append("repeats an earlier completion")
    return failed_checks


def print_evaluation_result(result: dict) -> None:
    failed_checks = failed_evaluation_checks(result)
    outcome = "✅" if result["successful"] else "❌"
    details = (
        "" if not failed_checks else f" ({', '.join(failed_checks)})"
    )
    print(
        f"{outcome} {result['seed']:>2}: "
        f"{result['response']!r}{details}"
    )


def report_evaluation_summary(results: list[dict]) -> None:
    attempts = len(results)
    successes = sum(result["successful"] for result in results)
    success_rate = successes / attempts if attempts else 0
    report_diversity(results)
    print(
        f"Successful completions: {successes} of {attempts} "
        f"({success_rate:.1%})"
    )

    failures = [result for result in results if not result["successful"]]
    if failures:
        print("Failures:")
        for result in failures:
            reasons = ", ".join(failed_evaluation_checks(result))
            print(f"Seed {result['seed']}: {reasons}")
    else:
        print("Failures: none")


def report_evaluation(results: list[dict]) -> None:
    for result in results:
        print_evaluation_result(result)
    report_evaluation_summary(results)

Apply the evaluator to the greedy completions from Part 1. Because greedy decoding produced the same otherwise-valid response ten times, only its first occurrence counts as a success. This makes the contrast between deterministic repetition and varied sampling concrete.

In [ ]:
greedy_evaluations = evaluate_completions(
    greedy_results,
    expected_prefix="The cat",
)
report_evaluation(greedy_evaluations)

Now try the evaluator on one non-greedy configuration: `top_k=20`, `temperature=0.7`, and `top_p=0.8`. Multiple tokens can survive at each step, so the seeds can now lead to different completions. Each seed's response is evaluated and displayed as soon as Ollama returns it; the diversity and success summaries appear after all ten generations finish.

In [ ]:
sampled_evaluations = generate_completions(
    USER_PROMPT,
    expected_prefix=USER_PROMPT,
    top_k= 20,
    temperature=0.7,
    top_p=0.8
)
report_evaluation_summary(sampled_evaluations)

## Part 3: Explore Sampling Parameters

Now use the interactive controls to change **one input at a time**. The prompt box begins with `The cat `, and you can edit it to try another sentence starter. You can also try a lower or higher `temperature`, a smaller or larger `top_k`, or a lower `top_p`. Before each run, predict whether the completions will become more or less varied. Keep the seeds and the other inputs fixed so you can attribute any observed difference to the input you changed.

Each run uses the Part 2 evaluator and prints its check or cross immediately after each response arrives. Compare both the number of unique responses and the success rate: variety and task success are different measurements.

In [ ]:
import ipywidgets as widgets

@widgets.interact_manual(
    user_prompt=widgets.Text(value="The cat ", description="Prompt"),
    top_k=widgets.IntSlider(value=25, min=1, max=100, step=1),
    temperature=widgets.FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05),
    top_p=widgets.FloatSlider(value=0.95, min=0.05, max=1.0, step=0.05),
)
def explore_sampling(user_prompt, top_k, temperature, top_p):
    if not user_prompt.strip():
        print("Enter a non-empty prompt.")
        return

    results = generate_completions(
        user_prompt,
        top_k=top_k,
        temperature=temperature,
        top_p=top_p,
        expected_prefix=user_prompt,
    )
    report_evaluation_summary(results)

## Challenges

1. Find parameters that pass 0 out of 10 evaluation checks.
2. Find parameters that pass 10 out of 10 evaluation checks.
3. Try changing the user prompt to "All cars " and finding parameters that pass at least 8 out of 10 evals. Can you identify any factually false statements along the way?

### This is a good moment to pause and reflect on a few bigger picture questions.

1. Do you have an intuitive feel for how the same prompt given to the same model can generate different results due to sampling?
2. Can you now explain one plausible source of where the phenomenon of _hallucination_ arises? (Hallucination is when an LLM "makes something up" which reads as reasonable but it actually made up or not true.)
3. Due to the _autoregressive_ nature of building a response based on all context before it, can you describe how some sampling parameters can cause an LLM to run astray and off topic?

## Part 4: Sampling for Structured Output

Sentence completion benefits from some variety: several different continuations can all be useful. Structured output changes the objective. A program consuming JSON needs the response to satisfy a precise contract, and there is little value in creative alternatives that violate it. A single unexpected token, such as an opening code fence, a trailing explanation, a missing quote, or the wrong value can make the result unusable. Program code has the same sensitivity to syntax, plus the additional requirement that it behave correctly when executed.

Our final task is to translate ten Python function calls into JSON. We are **not executing the calls**. For each textual input, the model must return exactly one JSON object with this shape:

```json
{"name": "name_of_function", "args": ["list", "of", "argument", "values"]}
```

The function name must be a string, `args` must preserve the positional arguments and their order, and Python's `True`, `False`, and `None` must become JSON's `true`, `false`, and `null`. The output must contain exactly the keys `name` and `args`, with no Markdown fence or explanatory prose.

This is deliberately a prompt-and-sampling experiment: we will parse the response with `json.loads`, but we will not ask Ollama to force JSON mode. Keep the model, system prompt, ten inputs, token limit, and seeds fixed while changing sampling parameters. Then a change in success rate belongs to the parameter configuration rather than a changing test.

In [ ]:
import json

JSON_SYSTEM_PROMPT = (
    "Convert the user's Python function call into exactly one JSON object. "
    'Use exactly two keys: "name" for the function name and "args" for '
    "a list of positional argument values in their original order. Convert "
    "Python True, False, and None to JSON true, false, and null. Respond with "
    "only the JSON object: no Markdown, code fence, or explanation."
)
JSON_MAX_GENERATED_TOKENS = 128
JSON_TEST_CASES = [
    {
        "call": 'greet("Ada")',
        "expected": {"name": "greet", "args": ["Ada"]},
    },
    {
        "call": "add(17, 25)",
        "expected": {"name": "add", "args": [17, 25]},
    },
    {
        "call": 'resize_image("cat.png", 800, 600)',
        "expected": {"name": "resize_image", "args": ["cat.png", 800, 600]},
    },
    {
        "call": "set_alarm(7, 30, True)",
        "expected": {"name": "set_alarm", "args": [7, 30, True]},
    },
    {
        "call": 'format_name("Jordan", "Lee", None)',
        "expected": {"name": "format_name", "args": ["Jordan", "Lee", None]},
    },
    {
        "call": "move_cursor(3.5, -2.25)",
        "expected": {"name": "move_cursor", "args": [3.5, -2.25]},
    },
    {
        "call": 'send_email("sam@example.com", "Hello!")',
        "expected": {"name": "send_email", "args": ["sam@example.com", "Hello!"]},
    },
    {
        "call": "clamp(12, 0, 10)",
        "expected": {"name": "clamp", "args": [12, 0, 10]},
    },
    {
        "call": 'tag_item("notebook", "school", "ai")',
        "expected": {"name": "tag_item", "args": ["notebook", "school", "ai"]},
    },
    {
        "call": 'record_score("blue", 98, False)',
        "expected": {"name": "record_score", "args": ["blue", 98, False]},
    },
]

for case_number, case in enumerate(JSON_TEST_CASES, start=1):
    expected_json = json.dumps(case["expected"], separators=(",", ":"))
    print(f"{case_number}: {case['call']} -> {expected_json}")

### Evaluate syntax, shape, and meaning

A response succeeds only if `json.loads` can parse it, its root is an object with exactly the keys `name` and `args`, and both values exactly match the expected function call. These checks separate three kinds of failure:

- **Syntax:** the response is not valid JSON, perhaps because it uses Python literals, Markdown, or prose.
- **Shape:** the JSON is valid but does not follow the two-key object contract.
- **Meaning:** the object has the wrong function name, argument order, values, or JSON value types.

We use `json.loads`, not `eval`, because parsing data should not execute it. The canonical comparison below also distinguishes values that Python sometimes considers equal, such as `false` and `0`. As before, each result prints immediately, followed by an aggregate failure summary.

In [ ]:
def json_values_match(actual, expected) -> bool:
    return json.dumps(
        actual, sort_keys=True, separators=(",", ":")
    ) == json.dumps(
        expected, sort_keys=True, separators=(",", ":")
    )


def evaluate_json_response(response: str, expected: dict) -> dict:
    try:
        parsed = json.loads(response)
    except json.JSONDecodeError as error:
        return {
            "parsed": None,
            "failure_reasons": [f"invalid JSON: {error.msg}"],
            "successful": False,
        }

    failure_reasons = []
    if type(parsed) is not dict:
        failure_reasons.append("JSON root is not an object")
    else:
        if set(parsed) != {"name", "args"}:
            failure_reasons.append("keys are not exactly name and args")
        if parsed.get("name") != expected["name"]:
            failure_reasons.append("incorrect function name")
        if not json_values_match(parsed.get("args"), expected["args"]):
            failure_reasons.append("incorrect argument values or types")

    return {
        "parsed": parsed,
        "failure_reasons": failure_reasons,
        "successful": not failure_reasons,
    }


def generate_json_call_results(
    *,
    top_k: int,
    temperature: float,
    top_p: float,
) -> list[dict]:
    results = []

    for case_number, case in enumerate(JSON_TEST_CASES, start=1):
        seed = FIRST_SEED + case_number - 1
        response = ollama.generate(
            model=MODEL,
            prompt=f"Python function call:\n{case['call']}",
            system=JSON_SYSTEM_PROMPT,
            think=False,
            options={
                "seed": seed,
                "num_predict": JSON_MAX_GENERATED_TOKENS,
                "top_k": top_k,
                "temperature": temperature,
                "top_p": top_p,
            },
        )

        generated_json = response.response.strip()
        evaluation = evaluate_json_response(
            generated_json, case["expected"]
        )
        result = {
            "case_number": case_number,
            "seed": seed,
            "call": case["call"],
            "response": generated_json,
            "generated_tokens": response.eval_count or 0,
            "duration_seconds": (response.total_duration or 0) / 1_000_000_000,
            **evaluation,
        }
        results.append(result)

        outcome = "✅" if result["successful"] else "❌"
        details = (
            ""
            if result["successful"]
            else f" ({', '.join(result['failure_reasons'])})"
        )
        print(
            f"{outcome} {case_number}: {case['call']} -> "
            f"{generated_json!r}{details}",
            flush=True,
        )

    return results


def report_json_summary(results: list[dict]) -> None:
    successes = sum(result["successful"] for result in results)
    attempts = len(results)
    success_rate = successes / attempts if attempts else 0
    print(
        f"Successful JSON conversions: {successes} of {attempts} "
        f"({success_rate:.1%})"
    )

    failures = [result for result in results if not result["successful"]]
    if failures:
        print("Failures:")
        for result in failures:
            reasons = ", ".join(result["failure_reasons"])
            print(f"Case {result['case_number']}: {reasons}")
    else:
        print("Failures: none")

### Experiment with the sampling controls

Press **Run Interact** to evaluate one parameter configuration across all ten calls. The defaults begin conservatively because this task rewards contract fidelity rather than novelty. Then change one parameter at a time and rerun. The ten calls and seeds stay fixed, so the resulting scores are directly comparable within this notebook.

Do not assume that one parameter setting is universally best. Record what this model actually produces under this prompt and evaluator. A setting that helps exact JSON may be unnecessarily repetitive for open-ended writing, while a setting that produces engaging prose may be fragile when downstream software expects a schema.

In [ ]:
@widgets.interact_manual(
    top_k=widgets.IntSlider(value=50, min=1, max=100, step=1),
    temperature=widgets.FloatSlider(value=2.5, min=0.05, max=5.0, step=0.05),
    top_p=widgets.FloatSlider(value=0.95, min=0.05, max=1.0, step=0.05),
)
def explore_json_sampling(top_k, temperature, top_p):
    results = generate_json_call_results(
        top_k=top_k,
        temperature=temperature,
        top_p=top_p,
    )
    report_json_summary(results)

### Challenges and connection to generated programs

1. Run the default parameters and look at the failures. Can you make a hypothesis on a different parameter setting to avoid this?
2. Does greedy work here? What is the least restrictive settings you can find that still score 10/10.
3. Can you return to the Session 1 program-generation notebook and choose sampling parameters that improve an evaluation without changing the prompt?

## Takeaway Message

This session zoomed in on a few common sampling parameters of LLMs, but the takeaways I hope you will have are much bigger picture. First, you should have a much better feel for how LLMs like ChatGPT, Claude, and Gemini are truly text completion algorithms which use autoregressive sampling to generate responses based on the model's pre-trained weights. All a trained model "knows" how to do is to read context and produce logits which represent the probabilities of next tokens based on the model's training set. With sampling, we saw how non-deterministic responses can emerge from an otherwise deterministic algorithm. With sampling parameters you can control how much variation you would like the system to leave to chance. For creative tasks, like writing poetry or stories, perhaps more variation is desireable. For specific, constrained outputs like producing well-formed JSON data or programming language syntax, less variation can improve results.

When using consumer-grade products from frontier labs, such as ChatGPT/Codex and Claude Code, these sampling parameters are chosen for you by the labs after extensive evaluations on what generally produces the best results. This is why you haven't needed to learn about these parameters when using these tools before and generally won't in the future. When building your own technology stacks that incorporate LLMs for various tasks, though, you will have these knobs available to you, in addition to system prompt and model size. You should intentionally evaluate your tasks' results with intuitively reasonable settings for these sampling parameters, especially if you aim to produce highly structured, predictable results such as converting between data formats. In many cases, you will find that tuning sampling parameters helps you achieve more predictable results than iterating on your prompt's instructions.